# Feature Engineering
特征工程这里用的是`train_cleaned.csv`，并不是preprocessing步骤的最终结果。因为onehot encoding之后不适合做特征工程。

首先，我们照例引入一下路径，并导入数据

In [1]:
from pathlib import Path

import pandas as pd

CURRENT_DIR = Path().cwd()
if CURRENT_DIR.name == "notebooks":
    #当前在notebooks/下面
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    #当前在根目录
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

df = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
# 先保存一个full features版本
df.to_csv(PROCESSED_DIR / "cleaned_full_features.csv", index=False)

我们接下来考虑特征。首先，我能想到这些特征：

* 年份可以用差值来代替建造时间

    GarageAge = YrSold - GarageYrBlt, 售卖年份减去车库建造年份，即售卖时，车库的年份

    HouseAge = YrSold - YearBuilt, 售卖年份减去建造年份，即售卖时房子的年份

    RemodAge = YrSold - YearRemodAdd, 售卖年份减去翻新年份，即售卖时距离上一次翻新的时间

* 可以把所有的面积加起来计算总使用面积

    TotalUtilSF = 1stFlrSF + 2ndFlrSF + TotalBsmtSF, 第一层加第二层加地下室总面积，即总建筑内可使用面积

* 卫生间等效加总

    EqvlBath = BsmtFullBath + 0.5 * BsmtHalfBath + FullBath + 0.5 * HalfBath

* 比例系列

    LivingRatio = GrLivArea/LotArea 可居住利用率

    BedroomRatio = Bedroom/TotalUtilSF单位使用面积的卧室数量

* 面积×质量

    OvrallMul = OverallQual * LotArea, 总体质量乘以面积

    BsmtMul = BsmtQual* TotalBsmtSF, 地下室质量乘以面积

    KitMul = Kitchen * KitchenQual, 厨房质量乘以面积

* Porch 总面积

    TotalPorchSF = OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch, 门廊总面积

* 设施存在性

    HasPool, HasGarage, HasBsmt, HasFireplace, Has2ndfloor, HasPorch, HasFence各个设施的存在性

* 总设施数量

    Totalutil = HasPool + HasGarage + HasBsmt + HasFireplace + Has2ndfloor + HasPorch + HasFence, 总设施数量

* 根号项

    RootUtil = TotalUtilSF^0.5
    RootLotArea = LotArea^0.5

接下来我们来尝试构造这些量

In [2]:
# * 年份可以用差值来代替建造时间
df["GarageAge"] = 0
mask = (df["HasGarage"] == 1)
df.loc[mask, "GarageAge"] = (df.loc[mask, "YrSold"] - df.loc[mask, "GarageYrBlt"])
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

# * 可以把所有的面积加起来计算总使用面积
df["TotalUtilSF"] = df["1stFlrSF"] + df["2ndFlrSF"] + df["TotalBsmtSF"]

# * 卫生间等效加总
df["EqvlBath"] = df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"] + df["FullBath"] + 0.5 * df["HalfBath"]

# * 比例系列
df["LivingRatio"] = df["GrLivArea"] / df["LotArea"]
df["BedroomRatio"] = df["Bedroom"] / df["TotalUtilSF"]

# * 面积×质量
df["OvrallMul"] = df["OverallQual"] * df["LotArea"]
df["BsmtMul"] = df["BsmtQual"] * df["TotalBsmtSF"]
df["KitMul"] = df["Kitchen"] * df["KitchenQual"] # 这个是数量 * 质量

# * Porch 总面积
df["TotalPorchSF"] = df["OpenPorchSF"] + df["EnclosedPorch"] + df["3SsnPorch"] + df["ScreenPorch"]

# * 设施存在性
# HasGarage已经有了
df["HasPool"] = 1*(df["PoolArea"] > 0)
df["HasBsmt"] = 1*(df["BsmtQual"] > 0)
df["HasFireplace"] = 1*(df["Fireplaces"] > 0)
df["Has2ndfloor"] = 1*(df["2ndFlrSF"] > 0)
df["HasPorch"] = 1*(df["TotalPorchSF"] > 0)
df["HasFence"] = 1*(df["Fence"] > 0 )

# * 总设施数量
df["Totalutil"] = df["HasPool"] + df["HasGarage"] + df["HasBsmt"] + df["HasFireplace"] + df["Has2ndfloor"] + df["HasPorch"] + df["HasFence"]

# * 根号项
df["RootUtil"] = df["Totalutil"] ** 0.5
df["RootLotArea"] = df["LotArea"] ** 0.5



In [3]:
df.to_csv(PROCESSED_DIR / "cleaned_manual_engineered.csv", index=False)

## Feature Selection

这一步我们直接应用选择算法来select feature，并且应用得到两版，最终应该得到三个feature set

Feature Set A：Full features, `cleaned_full_features.csv`（已保存）

Feature Set B：Statistical selection, `cleaned_stat_features.csv`

Feature Set C：Statistical + manual/domain selection, `cleaned_fully_engineered.csv`

不过，由于我们打算使用统计方法选择，而这个应该在编码后完成，所以我们决定直接把这一步放到Modelling里面去